# Chapter 16 Companion Notebook: Sequence and Attention Models in Business Analytics

**Book:** *Business Analytics and Artificial Intelligence: An Advanced Guide to Data-Driven Decision Making*  
**Book authors:** Hyunhwan "Aiden" Lee and Reo Song  
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/indy16mm/business-analytics-ai/blob/main/notebooks/Ch16_Sequence_and_Attention_Models.ipynb)

This notebook accompanies Chapter 16 of the book.

**License:** Use of this notebook is governed by the repository's
[Limited Companion Materials License](../LICENSE).




Classroom note: this notebook uses synthetic customer event data and small PyTorch models so students can run the workflow in Colab without a paid API or external dataset.

Copyright 2026 to present.

## How to use this notebook

Run the cells from top to bottom. Treat markdown sections as short lecture notes and code sections as live demos. If a section runs slowly, keep `FAST_MODE = True`, reduce `N_CUSTOMERS`, or skip the optional LSTM comparison.

## Why this matters (business framing)

Many business datasets look like ordinary tables after preprocessing, but the underlying evidence often arrives as a sequence. Customer journeys unfold across sessions, sales evolve over time, support tickets follow earlier failures, and fraud signals appear as event streams. A sequence model asks a different question from a static classifier: what did the model know at the decision moment, what pattern did it observe before that moment, and what outcome occurred after that moment?

This notebook follows the practical logic of Chapter 16. We will start with raw logs, define an as-of time, build lookback windows and prediction horizons, compare a feature baseline against recurrent and attention-based models, and end with a leakage audit.

## Agenda

1. Setup and reproducibility
2. Synthetic business event log
3. From logs to sequence examples: as-of time, lookback, and horizon
4. Baseline first: time-aware feature engineering with logistic regression
5. RNN, GRU, and LSTM intuition through a small sequence classifier
6. Attention pooling: selective access to past events
7. Self-attention and causal masking with a QKV heatmap
8. Tiny Transformer classifier for event sequences
9. Sequence length as a cost driver
10. Leakage audit and decision guide
11. Exercises and saved artifacts

## Learning objectives (measurable)

By the end of this notebook, you should be able to construct a time-honest sequence dataset, explain the difference between feature baselines and learned sequential representations, train a small GRU or LSTM classifier, inspect attention weights as a diagnostic lens, show how causal masking prevents future peeking, compare a tiny Transformer against simpler models, and run a leakage audit before trusting offline results.

## Connection map

You already covered classical feature engineering, supervised classification, embeddings, and the basic deep learning training workflow. This notebook adds order and decision time. The workflow becomes: define the decision contract, construct sequences without future information, build a simple baseline, train a small sequential model, inspect results, and check for leakage before making managerial claims.

In [ ]:
# ============================================================
# 1. Setup and reproducibility
# - install missing packages if needed
# - import libraries
# - set seeds
# - configure output folders
# ============================================================
import os
import sys
import math
import json
import time
import random
import warnings
import importlib
import subprocess
from pathlib import Path


def ensure(pkg_import_name, pip_name=None):
    """Install a package only if it is missing."""
    try:
        importlib.import_module(pkg_import_name)
    except Exception:
        pip_target = pip_name or pkg_import_name
        print(f"Installing {pip_target} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_target])


for import_name, pip_name in [
    ("numpy", None),
    ("pandas", None),
    ("sklearn", "scikit-learn"),
    ("matplotlib", None),
    ("tqdm", None),
    ("torch", None),
]:
    ensure(import_name, pip_name)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import display, Markdown

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 160)

SEED = 685
FAST_MODE = True

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# CPU thread guardrail. This prevents some notebook runtimes from becoming slow because of thread oversubscription.
if DEVICE == "cpu":
    torch.set_num_threads(1)

BASE_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
OUT_DIR = BASE_DIR / "baai_ch16_sequence_outputs"
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

if FAST_MODE:
    N_CUSTOMERS = 1800 if DEVICE == "cpu" else 2600
    TRAIN_EPOCHS = 4 if DEVICE == "cpu" else 5
else:
    N_CUSTOMERS = 3500 if DEVICE == "cpu" else 6000
    TRAIN_EPOCHS = 6

LOOKBACK_DAYS = 45
HORIZON_DAYS = 14
MAX_LEN = 32
BATCH_SIZE = 256 if DEVICE == "cpu" else 512

print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print({
    "FAST_MODE": FAST_MODE,
    "N_CUSTOMERS": N_CUSTOMERS,
    "TRAIN_EPOCHS": TRAIN_EPOCHS,
    "LOOKBACK_DAYS": LOOKBACK_DAYS,
    "HORIZON_DAYS": HORIZON_DAYS,
    "MAX_LEN": MAX_LEN,
    "OUT_DIR": str(OUT_DIR),
})

## Utility functions

We will reuse these helpers throughout the notebook. They keep the main sections focused on modeling decisions rather than display details.

In [ ]:
# ============================================================
# Utility functions
# ============================================================

def print_section(title):
    print("=" * len(title))
    print(title)
    print("=" * len(title))


def sigmoid_np(x):
    return 1 / (1 + np.exp(-x))


def safe_auc(y_true, prob):
    try:
        return roc_auc_score(y_true, prob)
    except Exception:
        return np.nan


def binary_metrics(y_true, prob, threshold=0.50):
    pred = (prob >= threshold).astype(int)
    return {
        "threshold": float(threshold),
        "accuracy": accuracy_score(y_true, pred),
        "f1": f1_score(y_true, pred, zero_division=0),
        "roc_auc": safe_auc(y_true, prob),
        "pred_positive_rate": float(pred.mean()),
    }


def find_best_threshold(y_true, prob, metric="f1"):
    thresholds = np.linspace(0.05, 0.95, 91)
    rows = []
    for th in thresholds:
        m = binary_metrics(y_true, prob, threshold=th)
        rows.append((th, m.get(metric, np.nan)))
    best_th, best_score = max(rows, key=lambda x: -np.inf if np.isnan(x[1]) else x[1])
    return float(best_th), float(best_score)


def plot_confusion_matrix(y_true, prob, threshold=0.50, title="Confusion matrix"):
    pred = (prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, pred)
    fig, ax = plt.subplots(figsize=(4.8, 4.2))
    im = ax.imshow(cm)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["no_future_cancel", "future_cancel"])
    ax.set_yticklabels(["no_future_cancel", "future_cancel"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(title)
    for (i, j), val in np.ndenumerate(cm):
        ax.text(j, i, int(val), ha="center", va="center")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()


def plot_history(history_df, title):
    if history_df is None or len(history_df) == 0:
        return
    plt.figure(figsize=(7, 4))
    plt.plot(history_df["epoch"], history_df["train_loss"], marker="o", label="train_loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title + " loss")
    plt.legend()
    plt.show()

    plt.figure(figsize=(7, 4))
    plt.plot(history_df["epoch"], history_df["roc_auc"], marker="o", label="val_auc")
    plt.plot(history_df["epoch"], history_df["f1"], marker="o", label="val_f1")
    plt.xlabel("Epoch")
    plt.ylabel("Score")
    plt.ylim(0, 1.05)
    plt.title(title + " validation metrics")
    plt.legend()
    plt.show()


print("Utility helpers ready.")

## 2. Synthetic business event log

To keep the notebook safe and shareable, we use synthetic customer event data. The simulated setting is a subscription business that wants to predict whether a customer will show a cancellation-intent event in the next two weeks. The input is the customer event history before the decision moment.

The synthetic data are designed to contain three teaching signals. First, recent friction events such as app errors and failed payments increase risk. Second, purchase activity reduces risk. Third, a small order-sensitive pattern is injected near the decision moment: the same events can imply different risk depending on whether friction follows checkout activity or appears earlier in the journey.

In [ ]:
# ============================================================
# 2.1 Synthetic event schema
# ============================================================
EVENT_NAMES = [
    "email_open",
    "site_visit",
    "product_view",
    "add_to_cart",
    "checkout_start",
    "purchase",
    "discount_view",
    "app_error",
    "support_ticket",
    "failed_payment",
    "cancel_intent",
]

EVENT_TO_ID = {name: i + 1 for i, name in enumerate(EVENT_NAMES)}
ID_TO_EVENT = {i: name for name, i in EVENT_TO_ID.items()}
ID_TO_EVENT[0] = "PAD"
TARGET_EVENT = "cancel_intent"

SEGMENTS = ["loyal", "browsing", "price_sensitive", "friction_risk"]
SEGMENT_PROBS = np.array([0.34, 0.30, 0.22, 0.14])

EVENT_PROBS = {
    "loyal": {
        "email_open": 0.18,
        "site_visit": 0.18,
        "product_view": 0.20,
        "add_to_cart": 0.10,
        "checkout_start": 0.08,
        "purchase": 0.18,
        "discount_view": 0.04,
        "app_error": 0.01,
        "support_ticket": 0.02,
        "failed_payment": 0.01,
    },
    "browsing": {
        "email_open": 0.16,
        "site_visit": 0.24,
        "product_view": 0.27,
        "add_to_cart": 0.08,
        "checkout_start": 0.04,
        "purchase": 0.05,
        "discount_view": 0.08,
        "app_error": 0.02,
        "support_ticket": 0.04,
        "failed_payment": 0.02,
    },
    "price_sensitive": {
        "email_open": 0.16,
        "site_visit": 0.20,
        "product_view": 0.22,
        "add_to_cart": 0.09,
        "checkout_start": 0.06,
        "purchase": 0.04,
        "discount_view": 0.16,
        "app_error": 0.02,
        "support_ticket": 0.03,
        "failed_payment": 0.02,
    },
    "friction_risk": {
        "email_open": 0.09,
        "site_visit": 0.18,
        "product_view": 0.18,
        "add_to_cart": 0.08,
        "checkout_start": 0.09,
        "purchase": 0.03,
        "discount_view": 0.09,
        "app_error": 0.09,
        "support_ticket": 0.10,
        "failed_payment": 0.07,
    },
}

for seg, d in EVENT_PROBS.items():
    s = sum(d.values())
    EVENT_PROBS[seg] = {k: v / s for k, v in d.items()}

pd.DataFrame({
    "event_id": [EVENT_TO_ID[e] for e in EVENT_NAMES],
    "event_name": EVENT_NAMES,
})

In [ ]:
# ============================================================
# 2.2 Generate synthetic raw logs
# ============================================================

def generate_synthetic_event_log(
    n_customers=N_CUSTOMERS,
    lookback_days=LOOKBACK_DAYS,
    horizon_days=HORIZON_DAYS,
    seed=SEED,
):
    rng = np.random.default_rng(seed)
    events = []
    decisions = []

    for cid in range(1, n_customers + 1):
        segment = str(rng.choice(SEGMENTS, p=SEGMENT_PROBS))
        as_of_day = int(rng.integers(70, 241))
        mean_events = {
            "loyal": 20,
            "browsing": 18,
            "price_sensitive": 22,
            "friction_risk": 24,
        }[segment]
        n_pre = int(np.clip(rng.poisson(mean_events), 6, 52))
        offsets = np.sort(rng.integers(1, lookback_days + 1, size=n_pre))[::-1]

        names = list(EVENT_PROBS[segment].keys())
        probs = np.array(list(EVENT_PROBS[segment].values()))
        pre_events = list(rng.choice(names, size=n_pre, p=probs))

        # Inject segment-specific motifs before computing the future outcome.
        if segment == "friction_risk" and rng.random() < 0.65:
            recent_positions = np.where(offsets <= 14)[0]
            if len(recent_positions) > 0:
                pre_events[int(rng.choice(recent_positions))] = str(
                    rng.choice(["app_error", "support_ticket", "failed_payment"], p=[0.45, 0.35, 0.20])
                )
        if segment == "price_sensitive" and rng.random() < 0.55:
            recent_positions = np.where(offsets <= 21)[0]
            if len(recent_positions) > 0:
                pre_events[int(rng.choice(recent_positions))] = "discount_view"
        if segment == "loyal" and rng.random() < 0.70:
            recent_positions = np.where(offsets <= 21)[0]
            if len(recent_positions) > 0:
                pre_events[int(rng.choice(recent_positions))] = "purchase"

        recent_events = [e for e, o in zip(pre_events, offsets) if o <= 14]
        very_recent_events = [e for e, o in zip(pre_events, offsets) if o <= 7]

        score = -2.10
        score += {"loyal": -0.70, "browsing": 0.00, "price_sensitive": 0.35, "friction_risk": 0.95}[segment]
        score += 0.95 * recent_events.count("failed_payment")
        score += 0.75 * recent_events.count("support_ticket")
        score += 0.55 * recent_events.count("app_error")
        score += 0.30 * recent_events.count("discount_view")
        score -= 0.45 * very_recent_events.count("purchase")
        future_cancel = int(rng.random() < sigmoid_np(score))

        # Order-sensitive motif near the decision moment.
        # Positive motif: checkout friction flows into support.
        # Negative motif: the same events occur in a less alarming order.
        if future_cancel == 1 and rng.random() < 0.65:
            offsets = np.concatenate([offsets, np.array([7, 4, 2])])
            pre_events = pre_events + ["checkout_start", "app_error", "support_ticket"]
        elif future_cancel == 0 and rng.random() < 0.35:
            offsets = np.concatenate([offsets, np.array([7, 4, 2])])
            pre_events = pre_events + ["support_ticket", "app_error", "checkout_start"]

        event_order = 0
        for e, offset in sorted(zip(pre_events, offsets), key=lambda x: as_of_day - x[1]):
            event_order += 1
            events.append({
                "customer_id": cid,
                "day": int(as_of_day - offset),
                "event_order": event_order,
                "event_name": e,
                "segment": segment,
                "as_of_day": as_of_day,
            })

        future_day = as_of_day + int(rng.integers(1, horizon_days + 1))
        if future_cancel == 1:
            future_name = TARGET_EVENT
        else:
            future_name = str(rng.choice(["email_open", "site_visit", "product_view", "purchase"], p=[0.30, 0.30, 0.25, 0.15]))

        events.append({
            "customer_id": cid,
            "day": future_day,
            "event_order": event_order + 1,
            "event_name": future_name,
            "segment": segment,
            "as_of_day": as_of_day,
        })

        decisions.append({
            "customer_id": cid,
            "as_of_day": as_of_day,
            "segment": segment,
            "latent_target_used_for_generation": future_cancel,
        })

    events_df = pd.DataFrame(events).sort_values(["customer_id", "day", "event_order"]).reset_index(drop=True)
    decisions_df = pd.DataFrame(decisions)
    return events_df, decisions_df


events_df, decisions_df = generate_synthetic_event_log()
print("events_df shape:", events_df.shape)
print("decisions_df shape:", decisions_df.shape)
display(events_df.head(12))
display(decisions_df.head())

In [ ]:
# ============================================================
# 2.3 Quick data checks
# ============================================================
print("Event counts:")
display(events_df["event_name"].value_counts().rename_axis("event_name").reset_index(name="count"))

print("Decision as-of day summary:")
display(decisions_df["as_of_day"].describe().to_frame("as_of_day"))

plt.figure(figsize=(7, 4))
plt.hist(decisions_df["as_of_day"], bins=24)
plt.title("Distribution of decision moments (as-of day)")
plt.xlabel("as-of day")
plt.ylabel("Number of examples")
plt.show()

## 3. From logs to sequence examples

A sequence example is defined by a decision contract. The unit is the customer, the as-of time is the day when the score is produced, the lookback window determines which history is allowed as input, and the prediction horizon determines how the label is computed. This is the most important habit in this notebook: inputs come from the past and the label comes from the future.

In [ ]:
# ============================================================
# 3.1 Visualize the prediction contract
# ============================================================
def plot_prediction_contract(lookback_days=LOOKBACK_DAYS, horizon_days=HORIZON_DAYS):
    fig, ax = plt.subplots(figsize=(9, 2.2))
    t0 = 0
    ax.hlines(0, -lookback_days, horizon_days, linewidth=3)
    ax.axvline(t0, linestyle="--", linewidth=2)
    ax.axvspan(-lookback_days, 0, alpha=0.20, label="input lookback window")
    ax.axvspan(0, horizon_days, alpha=0.20, label="future prediction horizon")
    ax.text(-lookback_days / 2, 0.10, "Inputs allowed", ha="center")
    ax.text(horizon_days / 2, 0.10, "Label observed", ha="center")
    ax.text(t0, -0.12, "as-of time t0", ha="center")
    ax.set_yticks([])
    ax.set_xlabel("Days relative to t0")
    ax.set_title("Lookback window and prediction horizon")
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0))
    plt.tight_layout()
    plt.show()

plot_prediction_contract()

In [ ]:
# ============================================================
# 3.2 Build padded event sequences
# ============================================================
RECENCY_BUCKET_NAMES = {
    0: "PAD",
    1: "0-1 days",
    2: "2-3 days",
    3: "4-7 days",
    4: "8-14 days",
    5: "15-30 days",
    6: "31+ days",
}


def recency_bucket(days_since_asof):
    if days_since_asof <= 1:
        return 1
    if days_since_asof <= 3:
        return 2
    if days_since_asof <= 7:
        return 3
    if days_since_asof <= 14:
        return 4
    if days_since_asof <= 30:
        return 5
    return 6


def build_sequence_examples(events_df, decisions_df, lookback_days=LOOKBACK_DAYS, horizon_days=HORIZON_DAYS, max_len=MAX_LEN):
    X_event = []
    X_recency = []
    lengths = []
    y = []
    rows = []

    grouped = {
        int(cid): g.sort_values(["day", "event_order"]).copy()
        for cid, g in events_df.groupby("customer_id")
    }

    for _, d in tqdm(decisions_df.iterrows(), total=len(decisions_df), desc="Building sequence examples"):
        cid = int(d["customer_id"])
        t0 = int(d["as_of_day"])
        g = grouped[cid]

        hist = g[(g["day"] <= t0) & (g["day"] > t0 - lookback_days)].copy()
        future = g[(g["day"] > t0) & (g["day"] <= t0 + horizon_days)].copy()
        label = int((future["event_name"] == TARGET_EVENT).any())

        if len(hist) > max_len:
            hist = hist.tail(max_len)

        event_ids = [EVENT_TO_ID[e] for e in hist["event_name"].tolist()]
        recency_ids = [recency_bucket(t0 - int(day)) for day in hist["day"].tolist()]
        length = len(event_ids)

        pad = max_len - length
        event_ids = event_ids + [0] * pad
        recency_ids = recency_ids + [0] * pad

        X_event.append(event_ids)
        X_recency.append(recency_ids)
        lengths.append(length)
        y.append(label)

        rows.append({
            "customer_id": cid,
            "as_of_day": t0,
            "segment": d["segment"],
            "label": label,
            "length": length,
            "sequence_text": " > ".join(hist["event_name"].tolist()),
            "future_events": " > ".join(future["event_name"].tolist()),
        })

    return (
        np.array(X_event, dtype=np.int64),
        np.array(X_recency, dtype=np.int64),
        np.array(lengths, dtype=np.int64),
        np.array(y, dtype=np.float32),
        pd.DataFrame(rows),
    )


X_event, X_recency, seq_lengths, y, example_df = build_sequence_examples(events_df, decisions_df)
print("X_event shape:", X_event.shape)
print("X_recency shape:", X_recency.shape)
print("label rate:", round(float(y.mean()), 3))
display(example_df.head(8))

In [ ]:
# ============================================================
# 3.3 Label distribution and a readable example
# ============================================================
summary_by_segment = (
    example_df.groupby("segment")
    .agg(n_examples=("label", "size"), future_cancel_rate=("label", "mean"), median_length=("length", "median"))
    .reset_index()
)
display(summary_by_segment)

sample_pos = example_df[example_df["label"] == 1].sample(1, random_state=SEED).iloc[0]
sample_neg = example_df[example_df["label"] == 0].sample(1, random_state=SEED).iloc[0]

print_section("Positive example")
print("Customer:", int(sample_pos["customer_id"]), "as-of day:", int(sample_pos["as_of_day"]))
print("History:", sample_pos["sequence_text"])
print("Future:", sample_pos["future_events"])

print()
print_section("Negative example")
print("Customer:", int(sample_neg["customer_id"]), "as-of day:", int(sample_neg["as_of_day"]))
print("History:", sample_neg["sequence_text"])
print("Future:", sample_neg["future_events"])

## 4. Baseline first: time-aware feature engineering

Before using a deep sequence model, build a simple baseline. Here the baseline turns the lookback window into interpretable features: event counts, recent event counts, sequence length, number of unique event types, and the last observed event. This baseline is fast, transparent, and often difficult to beat.

In [ ]:
# ============================================================
# 4.1 Time-based split
# ============================================================
train_idx = example_df.index[example_df["as_of_day"] <= 160].to_numpy()
val_idx = example_df.index[(example_df["as_of_day"] > 160) & (example_df["as_of_day"] <= 200)].to_numpy()
test_idx = example_df.index[example_df["as_of_day"] > 200].to_numpy()

split_rows = []
for name, idx in [("train", train_idx), ("val", val_idx), ("test", test_idx)]:
    split_rows.append({
        "split": name,
        "n": len(idx),
        "min_as_of_day": int(example_df.iloc[idx]["as_of_day"].min()),
        "max_as_of_day": int(example_df.iloc[idx]["as_of_day"].max()),
        "future_cancel_rate": float(y[idx].mean()),
    })

split_df = pd.DataFrame(split_rows)
display(split_df)

plt.figure(figsize=(8, 4))
for name, idx in [("train", train_idx), ("val", val_idx), ("test", test_idx)]:
    plt.hist(example_df.iloc[idx]["as_of_day"], bins=18, alpha=0.55, label=name)
plt.xlabel("as-of day")
plt.ylabel("Number of examples")
plt.title("Time-based split by decision moment")
plt.legend()
plt.show()

In [ ]:
# ============================================================
# 4.2 Build count and recency features
# ============================================================
def build_count_features(X_event, X_recency, lengths):
    recent_buckets = {1, 2, 3, 4}  # 0 to 14 days before t0
    rows = []
    for ev, rec, L in zip(X_event, X_recency, lengths):
        ev_real = ev[:L]
        rec_real = rec[:L]
        row = {
            "seq_length": float(L),
            "unique_event_types": float(len(set(ev_real.tolist()))) if L > 0 else 0.0,
            "last_event_id": float(ev_real[-1]) if L > 0 else 0.0,
        }
        for eid in range(1, len(EVENT_NAMES) + 1):
            event_name = ID_TO_EVENT[eid]
            row[f"count_{event_name}"] = float((ev_real == eid).sum())
            row[f"recent_count_{event_name}"] = float(((ev_real == eid) & np.isin(rec_real, list(recent_buckets))).sum())
        rows.append(row)
    return pd.DataFrame(rows)


X_tab = build_count_features(X_event, X_recency, seq_lengths)
print("Tabular baseline feature matrix:", X_tab.shape)
display(X_tab.head())

In [ ]:
# ============================================================
# 4.3 Train the baseline
# ============================================================
baseline = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED),
)
baseline.fit(X_tab.iloc[train_idx], y[train_idx])

val_prob_base = baseline.predict_proba(X_tab.iloc[val_idx])[:, 1]
test_prob_base = baseline.predict_proba(X_tab.iloc[test_idx])[:, 1]
base_threshold, base_val_f1 = find_best_threshold(y[val_idx], val_prob_base)
base_metrics = binary_metrics(y[test_idx], test_prob_base, threshold=base_threshold)

model_results = []
model_results.append({"model": "time_aware_logistic_baseline", **base_metrics})

print("Best validation threshold:", round(base_threshold, 3), "validation F1:", round(base_val_f1, 3))
print("Test metrics:")
display(pd.DataFrame([base_metrics]))
plot_confusion_matrix(y[test_idx], test_prob_base, threshold=base_threshold, title="Baseline test confusion matrix")

## 5. RNN, GRU, and LSTM classifiers

A recurrent model reads the sequence step by step and updates an internal state. A plain RNN is the simplest version. A GRU or LSTM adds gates that help the model decide what to retain and what to update. In business terms, the model is learning a compact summary of the customer state as events arrive.

We will use event embeddings and recency-bucket embeddings. This keeps the input close to real business logs, where each step contains categorical events and timing information.

In [ ]:
# ============================================================
# 5.1 PyTorch Dataset and DataLoaders
# ============================================================
class SequenceDataset(Dataset):
    def __init__(self, X_event, X_recency, lengths, y, indices):
        self.X_event = torch.as_tensor(X_event[indices], dtype=torch.long)
        self.X_recency = torch.as_tensor(X_recency[indices], dtype=torch.long)
        self.lengths = torch.as_tensor(lengths[indices], dtype=torch.long)
        self.y = torch.as_tensor(y[indices], dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_event[idx], self.X_recency[idx], self.lengths[idx], self.y[idx]


def make_loaders(batch_size=BATCH_SIZE):
    train_loader = DataLoader(
        SequenceDataset(X_event, X_recency, seq_lengths, y, train_idx),
        batch_size=batch_size,
        shuffle=True,
    )
    val_loader = DataLoader(
        SequenceDataset(X_event, X_recency, seq_lengths, y, val_idx),
        batch_size=batch_size,
        shuffle=False,
    )
    test_loader = DataLoader(
        SequenceDataset(X_event, X_recency, seq_lengths, y, test_idx),
        batch_size=batch_size,
        shuffle=False,
    )
    return train_loader, val_loader, test_loader


train_loader, val_loader, test_loader = make_loaders()
print("Batches:", {"train": len(train_loader), "val": len(val_loader), "test": len(test_loader)})

In [ ]:
# ============================================================
# 5.2 Model classes
# ============================================================
class RNNClassifier(nn.Module):
    def __init__(
        self,
        n_events,
        n_recency,
        rnn_type="GRU",
        event_dim=24,
        recency_dim=8,
        hidden_dim=48,
        dropout=0.20,
    ):
        super().__init__()
        self.rnn_type = rnn_type
        self.event_emb = nn.Embedding(n_events + 1, event_dim, padding_idx=0)
        self.recency_emb = nn.Embedding(n_recency + 1, recency_dim, padding_idx=0)
        input_dim = event_dim + recency_dim
        rnn_cls = {"RNN": nn.RNN, "GRU": nn.GRU, "LSTM": nn.LSTM}[rnn_type]
        self.rnn = rnn_cls(input_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, event_ids, recency_ids, lengths):
        x = torch.cat([self.event_emb(event_ids), self.recency_emb(recency_ids)], dim=-1)
        out, hidden = self.rnn(x)
        idx = (lengths - 1).clamp(min=0)
        batch_idx = torch.arange(out.size(0), device=out.device)
        final_state = out[batch_idx, idx]
        return self.fc(self.dropout(final_state)).squeeze(-1)


class AttentionGRUClassifier(nn.Module):
    def __init__(self, n_events, n_recency, event_dim=24, recency_dim=8, hidden_dim=48, dropout=0.20):
        super().__init__()
        self.event_emb = nn.Embedding(n_events + 1, event_dim, padding_idx=0)
        self.recency_emb = nn.Embedding(n_recency + 1, recency_dim, padding_idx=0)
        self.gru = nn.GRU(event_dim + recency_dim, hidden_dim, batch_first=True)
        self.attn = nn.Linear(hidden_dim, 1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, event_ids, recency_ids, lengths, return_attention=False):
        x = torch.cat([self.event_emb(event_ids), self.recency_emb(recency_ids)], dim=-1)
        out, _ = self.gru(x)
        mask = event_ids.ne(0)
        scores = self.attn(torch.tanh(out)).squeeze(-1)
        scores = scores.masked_fill(~mask, -1e9)
        weights = torch.softmax(scores, dim=1)
        context = (out * weights.unsqueeze(-1)).sum(dim=1)
        logits = self.fc(self.dropout(context)).squeeze(-1)
        if return_attention:
            return logits, weights
        return logits


print("Model classes ready.")

In [ ]:
# ============================================================
# 5.3 Training and prediction helpers
# ============================================================
@torch.no_grad()
def predict_torch(model, loader):
    model.eval()
    probs = []
    ys = []
    for event_ids, recency_ids, lengths, labels in loader:
        event_ids = event_ids.to(DEVICE)
        recency_ids = recency_ids.to(DEVICE)
        lengths = lengths.to(DEVICE)
        logits = model(event_ids, recency_ids, lengths)
        probs.append(torch.sigmoid(logits).detach().cpu().numpy())
        ys.append(labels.numpy())
    return np.concatenate(probs), np.concatenate(ys)


def train_torch_model(model, train_loader, val_loader, epochs=TRAIN_EPOCHS, lr=1e-3, name="model"):
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    pos_weight_value = float((1 - y[train_idx]).sum() / max(y[train_idx].sum(), 1))
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight_value, device=DEVICE))
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        n_seen = 0
        for event_ids, recency_ids, lengths, labels in train_loader:
            event_ids = event_ids.to(DEVICE)
            recency_ids = recency_ids.to(DEVICE)
            lengths = lengths.to(DEVICE)
            labels = labels.to(DEVICE)

            optimizer.zero_grad()
            logits = model(event_ids, recency_ids, lengths)
            loss = criterion(logits, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += loss.item() * len(labels)
            n_seen += len(labels)

        val_prob, val_y = predict_torch(model, val_loader)
        metrics = binary_metrics(val_y, val_prob, threshold=0.50)
        row = {"model": name, "epoch": epoch, "train_loss": total_loss / max(n_seen, 1), **metrics}
        history.append(row)
        print(
            f"{name} epoch {epoch:02d} | train_loss={row['train_loss']:.4f} | "
            f"val_auc={row['roc_auc']:.3f} | val_f1={row['f1']:.3f}"
        )

    return pd.DataFrame(history)


def evaluate_model_on_test(model, name):
    val_prob, val_y = predict_torch(model, val_loader)
    test_prob, test_y = predict_torch(model, test_loader)
    best_threshold, val_f1 = find_best_threshold(val_y, val_prob)
    metrics = binary_metrics(test_y, test_prob, threshold=best_threshold)
    metrics["val_selected_f1"] = val_f1
    print_section(name + " test metrics")
    print("Selected threshold from validation:", round(best_threshold, 3))
    display(pd.DataFrame([metrics]))
    return metrics, test_prob, test_y


print("Training helpers ready.")

In [ ]:
# ============================================================
# 5.4 Train GRU classifier
# ============================================================
gru_model = RNNClassifier(
    n_events=len(EVENT_NAMES),
    n_recency=max(RECENCY_BUCKET_NAMES.keys()),
    rnn_type="GRU",
)

gru_history = train_torch_model(gru_model, train_loader, val_loader, epochs=TRAIN_EPOCHS, lr=1e-3, name="GRU")
plot_history(gru_history, "GRU")

gru_metrics, gru_test_prob, torch_test_y = evaluate_model_on_test(gru_model, "GRU")
model_results.append({"model": "GRU", **gru_metrics})
plot_confusion_matrix(torch_test_y, gru_test_prob, threshold=gru_metrics["threshold"], title="GRU test confusion matrix")

In [ ]:
# ============================================================
# 5.5 Optional LSTM comparison
# - Set RUN_LSTM_COMPARISON = False if runtime is tight.
# ============================================================
RUN_LSTM_COMPARISON = True

if RUN_LSTM_COMPARISON:
    lstm_model = RNNClassifier(
        n_events=len(EVENT_NAMES),
        n_recency=max(RECENCY_BUCKET_NAMES.keys()),
        rnn_type="LSTM",
    )
    lstm_history = train_torch_model(lstm_model, train_loader, val_loader, epochs=TRAIN_EPOCHS, lr=1e-3, name="LSTM")
    plot_history(lstm_history, "LSTM")
    lstm_metrics, lstm_test_prob, _ = evaluate_model_on_test(lstm_model, "LSTM")
    model_results.append({"model": "LSTM", **lstm_metrics})
else:
    print("LSTM comparison skipped.")

## 6. Attention pooling: selective access to past events

A recurrent model compresses the past into a state. Attention pooling lets the model form a context vector by weighting the hidden state at each time step. This is useful when a few earlier events are more diagnostic than the final state alone. The weights are not a full explanation of causality, but they are a useful diagnostic lens for classroom inspection.

In [ ]:
# ============================================================
# 6.1 Train GRU with attention pooling
# ============================================================
attn_model = AttentionGRUClassifier(
    n_events=len(EVENT_NAMES),
    n_recency=max(RECENCY_BUCKET_NAMES.keys()),
)

attn_history = train_torch_model(attn_model, train_loader, val_loader, epochs=TRAIN_EPOCHS, lr=1e-3, name="GRU_attention")
plot_history(attn_history, "GRU with attention pooling")

attn_metrics, attn_test_prob, _ = evaluate_model_on_test(attn_model, "GRU with attention pooling")
model_results.append({"model": "GRU_attention_pooling", **attn_metrics})

In [ ]:
# ============================================================
# 6.2 Inspect attention weights for one held-out customer
# ============================================================
def inspect_attention_for_example(example_index):
    attn_model.eval()
    event_ids = torch.as_tensor(X_event[example_index:example_index + 1], dtype=torch.long).to(DEVICE)
    recency_ids = torch.as_tensor(X_recency[example_index:example_index + 1], dtype=torch.long).to(DEVICE)
    lengths_t = torch.as_tensor(seq_lengths[example_index:example_index + 1], dtype=torch.long).to(DEVICE)
    with torch.no_grad():
        logits, weights = attn_model(event_ids, recency_ids, lengths_t, return_attention=True)
        prob = torch.sigmoid(logits).item()
    L = int(seq_lengths[example_index])
    ev = X_event[example_index, :L]
    rec = X_recency[example_index, :L]
    w = weights[0, :L].detach().cpu().numpy()
    out = pd.DataFrame({
        "step": np.arange(1, L + 1),
        "event": [ID_TO_EVENT[int(e)] for e in ev],
        "recency_bucket": [RECENCY_BUCKET_NAMES[int(r)] for r in rec],
        "attention_weight": w,
    })
    return prob, out

positive_test_indices = [idx for idx in test_idx if y[idx] == 1]
example_index = int(positive_test_indices[0]) if positive_test_indices else int(test_idx[0])
prob, attn_df = inspect_attention_for_example(example_index)

print("Example index:", example_index)
print("True label:", int(y[example_index]), "predicted probability:", round(prob, 3))
display(example_df.loc[[example_index], ["customer_id", "as_of_day", "segment", "label", "sequence_text", "future_events"]])
display(attn_df.tail(14))

plt.figure(figsize=(8, 5))
plot_df = attn_df.tail(14).copy()
labels = [f"{int(r.step)}. {r.event}" for _, r in plot_df.iterrows()]
plt.barh(labels, plot_df["attention_weight"])
plt.xlabel("Attention weight")
plt.title("Attention weights for the most recent events")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 7. Self-attention and causal masking

Self-attention lets each position compare itself with other positions in the same sequence. The query, key, and value metaphor is a retrieval metaphor: the current position asks a question, other positions offer matching signatures, and the values provide the content to combine. For causal business tasks, future positions must be masked so the model cannot peek ahead.

In [ ]:
# ============================================================
# 7.1 Manual scaled dot-product attention demo
# ============================================================
def softmax_rows(scores):
    scores = scores - np.max(scores, axis=1, keepdims=True)
    exp_scores = np.exp(scores)
    return exp_scores / exp_scores.sum(axis=1, keepdims=True)


def qkv_attention_demo(tokens, causal=False, seed=SEED):
    rng = np.random.default_rng(seed)
    d_model = 16
    emb_table = rng.normal(0, 1, size=(len(EVENT_NAMES) + 1, d_model))
    ids = np.array([EVENT_TO_ID[t] for t in tokens])
    X = emb_table[ids]

    Wq = rng.normal(0, 0.30, size=(d_model, d_model))
    Wk = rng.normal(0, 0.30, size=(d_model, d_model))
    Wv = rng.normal(0, 0.30, size=(d_model, d_model))
    Q = X @ Wq
    K = X @ Wk
    V = X @ Wv

    scores = Q @ K.T / math.sqrt(d_model)
    if causal:
        future_mask = np.triu(np.ones_like(scores, dtype=bool), k=1)
        scores = scores.copy()
        scores[future_mask] = -1e9
    weights = softmax_rows(scores)
    context = weights @ V
    return weights, context


demo_tokens = [
    "email_open",
    "site_visit",
    "product_view",
    "checkout_start",
    "app_error",
    "support_ticket",
    "site_visit",
    "product_view",
]

bidirectional_weights, _ = qkv_attention_demo(demo_tokens, causal=False)
causal_weights, _ = qkv_attention_demo(demo_tokens, causal=True)

print("Demo sequence:")
for i, token in enumerate(demo_tokens, 1):
    print(f"{i:02d}. {token}")

In [ ]:
# ============================================================
# 7.2 Heatmaps: bidirectional versus causal attention
# ============================================================
def plot_attention_heatmap(weights, tokens, title):
    labels = [f"{i+1}:{tok}" for i, tok in enumerate(tokens)]
    plt.figure(figsize=(8, 6))
    plt.imshow(weights, aspect="auto")
    plt.xticks(range(len(tokens)), labels, rotation=90)
    plt.yticks(range(len(tokens)), labels)
    plt.xlabel("Key position attended to")
    plt.ylabel("Query position being updated")
    plt.title(title)
    plt.colorbar(label="attention weight")
    plt.tight_layout()
    plt.show()

plot_attention_heatmap(bidirectional_weights, demo_tokens, "Bidirectional self-attention")
plot_attention_heatmap(causal_weights, demo_tokens, "Causal self-attention with future mask")

## 8. Tiny Transformer classifier

A Transformer encoder uses self-attention to mix context across positions. Unlike recurrence, it does not automatically know order, so we add position embeddings. We also use a padding mask so the model ignores artificial padding positions.

In [ ]:
# ============================================================
# 8.1 Tiny Transformer encoder classifier
# ============================================================
class TinyTransformerClassifier(nn.Module):
    def __init__(
        self,
        n_events,
        n_recency,
        max_len,
        d_model=48,
        n_heads=4,
        ff_dim=96,
        n_layers=1,
        dropout=0.20,
    ):
        super().__init__()
        self.event_emb = nn.Embedding(n_events + 1, d_model, padding_idx=0)
        self.recency_emb = nn.Embedding(n_recency + 1, d_model, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, d_model)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(d_model, 1)

    def forward(self, event_ids, recency_ids, lengths):
        B, T = event_ids.shape
        positions = torch.arange(T, device=event_ids.device).unsqueeze(0).expand(B, T)
        x = self.event_emb(event_ids) + self.recency_emb(recency_ids) + self.pos_emb(positions)
        pad_mask = event_ids.eq(0)
        h = self.encoder(x, src_key_padding_mask=pad_mask)
        real_mask = (~pad_mask).float().unsqueeze(-1)
        pooled = (h * real_mask).sum(dim=1) / real_mask.sum(dim=1).clamp(min=1.0)
        return self.fc(self.dropout(pooled)).squeeze(-1)


transformer_model = TinyTransformerClassifier(
    n_events=len(EVENT_NAMES),
    n_recency=max(RECENCY_BUCKET_NAMES.keys()),
    max_len=MAX_LEN,
)

transformer_history = train_torch_model(transformer_model, train_loader, val_loader, epochs=TRAIN_EPOCHS, lr=1e-3, name="tiny_transformer")
plot_history(transformer_history, "Tiny Transformer")

transformer_metrics, transformer_test_prob, _ = evaluate_model_on_test(transformer_model, "Tiny Transformer")
model_results.append({"model": "tiny_transformer", **transformer_metrics})

In [ ]:
# ============================================================
# 8.2 Model comparison table
# ============================================================
model_results_df = pd.DataFrame(model_results)
metric_cols = ["model", "threshold", "accuracy", "f1", "roc_auc", "pred_positive_rate", "val_selected_f1"]
for col in metric_cols:
    if col not in model_results_df.columns:
        model_results_df[col] = np.nan
model_results_df = model_results_df[metric_cols].sort_values("roc_auc", ascending=False).reset_index(drop=True)
display(model_results_df)

plt.figure(figsize=(8, 4))
plt.bar(model_results_df["model"], model_results_df["roc_auc"])
plt.xticks(rotation=30, ha="right")
plt.ylim(0, 1.05)
plt.ylabel("Test ROC-AUC")
plt.title("Sequence model comparison")
plt.tight_layout()
plt.show()

## 9. Sequence length as a cost driver

Architecture choice is also a cost choice. Recurrent models process steps sequentially, while standard self-attention compares many pairs of positions. This does not mean one family is always better. It means sequence length and serving pattern should be treated as design variables.

In [ ]:
# ============================================================
# 9.1 Relative cost intuition
# ============================================================
seq_len_grid = np.arange(10, 401, 10)
cost_df = pd.DataFrame({
    "sequence_length": seq_len_grid,
    "rnn_relative_steps": seq_len_grid / seq_len_grid.max(),
    "attention_relative_pairs": (seq_len_grid ** 2) / (seq_len_grid.max() ** 2),
})

display(cost_df.head())

plt.figure(figsize=(7, 4))
plt.plot(cost_df["sequence_length"], cost_df["rnn_relative_steps"], label="RNN-style sequential steps")
plt.plot(cost_df["sequence_length"], cost_df["attention_relative_pairs"], label="standard self-attention pair comparisons")
plt.xlabel("Sequence length")
plt.ylabel("Relative cost scale")
plt.title("Sequence length as a modeling and serving cost driver")
plt.legend()
plt.show()

## 10. Leakage audit

Now we intentionally break the rules. The leaky feature below counts the target event in the future horizon and gives it to the model as an input. The result should look excellent, but it is not predictive skill. It is a warning sign. This cell is included because sequence projects often fail when future information quietly enters preprocessing, aggregation, or splitting.

In [ ]:
# ============================================================
# 10.1 Deliberate leakage demo
# ============================================================
X_leaky = X_tab.copy()
X_leaky["leaky_future_cancel_in_horizon"] = y  # Do not do this in a real model.

leaky_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED),
)
leaky_model.fit(X_leaky.iloc[train_idx], y[train_idx])
leaky_val_prob = leaky_model.predict_proba(X_leaky.iloc[val_idx])[:, 1]
leaky_test_prob = leaky_model.predict_proba(X_leaky.iloc[test_idx])[:, 1]
leaky_threshold, leaky_val_f1 = find_best_threshold(y[val_idx], leaky_val_prob)
leaky_metrics = binary_metrics(y[test_idx], leaky_test_prob, threshold=leaky_threshold)

leakage_demo_df = pd.DataFrame([
    {"model": "safe_baseline", **base_metrics},
    {"model": "leaky_baseline_with_future_target", **leaky_metrics},
])
display(leakage_demo_df)

plt.figure(figsize=(6, 4))
plt.bar(leakage_demo_df["model"], leakage_demo_df["roc_auc"])
plt.xticks(rotation=25, ha="right")
plt.ylim(0, 1.05)
plt.ylabel("Test ROC-AUC")
plt.title("Leakage can masquerade as model improvement")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 10.2 Sequence leakage audit checklist
# ============================================================
leakage_audit = pd.DataFrame([
    {
        "audit_question": "Are all inputs computed using only data at or before t0?",
        "safe_pattern": "Every feature window is anchored at the as-of time.",
        "failure_pattern": "Aggregation accidentally reaches into the future horizon.",
    },
    {
        "audit_question": "Are preprocessing steps fit only on the training period?",
        "safe_pattern": "Scaling and encoding are fit on training data and applied forward.",
        "failure_pattern": "Global normalization uses future distribution shifts.",
    },
    {
        "audit_question": "Does the split respect time order?",
        "safe_pattern": "Training precedes validation and test periods.",
        "failure_pattern": "Random splits mix earlier and later examples.",
    },
    {
        "audit_question": "Is the horizon respected and censoring handled?",
        "safe_pattern": "Examples with unobserved future horizons are excluded.",
        "failure_pattern": "Late-period examples are incorrectly treated as negatives.",
    },
    {
        "audit_question": "Are proxy future signals removed?",
        "safe_pattern": "Post-decision events and approvals are excluded from inputs.",
        "failure_pattern": "Operational outcomes become predictive features.",
    },
])

display(leakage_audit)

## 11. Decision guide and quick exercises

A practical sequence workflow starts with the decision problem, not with the architecture. If a time-aware baseline is strong and the sequence length is short, a simple model may be enough. If the effective context is moderate and streaming updates matter, a GRU or LSTM is a strong baseline. If the task requires flexible retrieval across distant positions and batch scoring is acceptable, a Transformer-style model may be worth the extra cost.

In [ ]:
# ============================================================
# 11.1 Save artifacts
# ============================================================
example_df.to_csv(OUT_DIR / "ch16_sequence_examples.csv", index=False)
model_results_df.to_csv(OUT_DIR / "ch16_model_results.csv", index=False)
leakage_audit.to_csv(OUT_DIR / "ch16_leakage_audit.csv", index=False)

print("Saved artifacts:")
print(" -", OUT_DIR / "ch16_sequence_examples.csv")
print(" -", OUT_DIR / "ch16_model_results.csv")
print(" -", OUT_DIR / "ch16_leakage_audit.csv")

## Exercises

1. Change `LOOKBACK_DAYS` from 45 to 21 or 60, rerun the notebook, and compare the baseline and GRU results. What changes when the model can see less or more history?

2. Change `MAX_LEN` from 32 to 16 or 64. Does truncating the sequence hurt the model, and does the runtime change?

3. Turn `RUN_LSTM_COMPARISON` off and on. Is the LSTM meaningfully better than the GRU in this synthetic setting?

4. Remove the recency embedding from the RNN model. What happens when the model sees event order but not timing buckets?

5. Find a false positive from the test set and inspect the attention weights. Which events appear to drive the score, and would a human reviewer agree that those events are risky?

6. Create one additional leakage check. For example, add a rule that flags feature names containing `future`, `after_t0`, `horizon`, or `post_decision`.

## Wrap-up

This notebook showed the full sequence workflow: raw logs, as-of time, lookback window, horizon label, time-aware baseline, recurrent model, attention pooling, self-attention masking, Transformer classifier, and leakage audit. The main lesson is stable across tools: sequence construction and evaluation discipline matter as much as the architecture.